# API Fetching Notebook

## Import 

In [1]:
import os
import time
import requests
import json
import pandas as pd
from datetime import datetime, timezone, timedelta

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Config

In [2]:
WALLETS = {
    "rn1":      "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
    "sovereign": "0xee613b3fc183ee44f9da9c05f53e2da107e3debf",
    "swisstony": "0x204f72f35326db932158cba6adff0b9a1da95e14",
}

DATA_API  = "https://data-api.polymarket.com"   # positions, trades, activity
GAMMA_API = "https://gamma-api.polymarket.com"  # profiles, market metadata

DATA_DIR    = "data"
LEDGER_FILE = f"{DATA_DIR}/ledger.json"
os.makedirs(DATA_DIR, exist_ok=True)

## Ledger Setup

In [3]:
def load_ledger():
    if os.path.exists(LEDGER_FILE):
        with open(LEDGER_FILE) as f:
            return json.load(f)
    return {name: [] for name in WALLETS}

def save_ledger(ledger):
    with open(LEDGER_FILE, "w") as f:
        json.dump(ledger, f, indent=2)

def window_done(ledger, wallet_name, win_start, win_end):
    """Check if this exact window is already in the ledger."""
    return [win_start, win_end] in ledger.get(wallet_name, [])

def mark_window_done(ledger, wallet_name, win_start, win_end):
    """Record a completed window in the ledger and flush to disk."""
    ledger.setdefault(wallet_name, []).append([win_start, win_end])
    save_ledger(ledger)

ledger = load_ledger()
print("Ledger loaded:", {k: len(v) for k, v in ledger.items()}, "windows completed")

Ledger loaded: {'rn1': 42, 'sovereign': 38, 'swisstony': 37} windows completed


## HTTP Caller

In [4]:
def get(base_url, endpoint, params=None, retries=3, backoff=2.0):
    """
    GET request with simple retry on transient connection errors.

    retries: number of attempts before giving up
    backoff: seconds to wait between attempts (doubles each retry)
    """
    url = f"{base_url}{endpoint}"
    wait = backoff

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text}")
            return response.json()

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt == retries:
                raise  # give up after final attempt
            print(f"  ⚠ connection error (attempt {attempt}/{retries}), retrying in {wait}s...")
            time.sleep(wait)
            wait *= 2  # exponential backoff: 2s → 4s → 8s

## Fetch Exploration

### Position

In [5]:
positions = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/positions", params={"user": address, "limit": 500})
    positions[name] = data
    print(f"\n=== {name} — {len(data)} open positions ===")
    if data:
        print("Fields:", list(data[0].keys()))
        print("First record:")
        print(json.dumps(data[0], indent=2))


=== rn1 — 500 open positions ===
Fields: ['proxyWallet', 'asset', 'conditionId', 'size', 'avgPrice', 'initialValue', 'currentValue', 'cashPnl', 'percentPnl', 'totalBought', 'realizedPnl', 'percentRealizedPnl', 'curPrice', 'redeemable', 'mergeable', 'title', 'slug', 'icon', 'eventId', 'eventSlug', 'outcome', 'outcomeIndex', 'oppositeOutcome', 'oppositeAsset', 'endDate', 'negativeRisk']
First record:
{
  "proxyWallet": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
  "asset": "1085310679959577834629905872167609894989809542716168749768240119209698690157",
  "conditionId": "0xd57db42dc26f8cdd6f26e9148ba5b40775af15ea104b6d0068ebfd4c7342d7aa",
  "size": 168767.8979,
  "avgPrice": 0.3264,
  "initialValue": 55090.7361,
  "currentValue": 22783.6662,
  "cashPnl": -32307.0699,
  "percentPnl": -58.6433,
  "totalBought": 169734.0224,
  "realizedPnl": 0,
  "percentRealizedPnl": -58.8787,
  "curPrice": 0.135,
  "redeemable": false,
  "mergeable": true,
  "title": "Rockets vs. Lakers",
  "slug": "nba-

### Values

In [6]:
values = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/value", params={"user": address})
    values[name] = data
    print(f"{name}: {json.dumps(data, indent=2)}")

rn1: [
  {
    "user": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
    "value": 547960.1841
  }
]
sovereign: [
  {
    "user": "0xee613b3fc183ee44f9da9c05f53e2da107e3debf",
    "value": 26881.2012
  }
]
swisstony: [
  {
    "user": "0x204f72f35326db932158cba6adff0b9a1da95e14",
    "value": 583961.2258
  }
]


### Activity

In [7]:
activity_sample = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/activity", params={
        "user":          address,
        "limit":         10,           # small on purpose — just inspecting shape
        "sortDirection": "DESC",       # most recent first
    })
    activity_sample[name] = data
    print(f"\n=== {name} — showing {len(data)} records ===")
    if data:
        print("Fields:", list(data[0].keys()))
        print("Most recent event:")
        print(json.dumps(data[0], indent=2))
        print(f"Oldest in this page:")
        print(json.dumps(data[-1], indent=2))


=== rn1 — showing 10 records ===
Fields: ['proxyWallet', 'timestamp', 'conditionId', 'type', 'size', 'usdcSize', 'transactionHash', 'price', 'asset', 'side', 'outcomeIndex', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized']
Most recent event:
{
  "proxyWallet": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
  "timestamp": 1776834846,
  "conditionId": "0xd57db42dc26f8cdd6f26e9148ba5b40775af15ea104b6d0068ebfd4c7342d7aa",
  "type": "TRADE",
  "size": 27.6,
  "usdcSize": 26.772,
  "transactionHash": "0x55f5ebebc817457bdae312e378d536f91f80f59eb6674796046da6353753018d",
  "price": 0.97,
  "asset": "45949080565643676028230495363401189884973675894037073023779613649229440981535",
  "side": "BUY",
  "outcomeIndex": 1,
  "title": "Rockets vs. Lakers",
  "slug": "nba-hou-lal-2026-04-21",
  "icon": "https://polymarket-upload.s3.us-east-2.amazonaws.com/super+cool+basketball+in+red+and+blue+wow.png",
  "eventSlug": "nba-hou-lal-2026-

### Earliest Activity

In [8]:
for name, address in WALLETS.items():
    data = get(DATA_API, "/activity", params={
        "user":          address,
        "limit":         1,
        "sortDirection": "ASC",   # flip to oldest first
    })
    if data:
        ts = data[0]["timestamp"]
        dt = datetime.fromtimestamp(ts, tz=timezone.utc)
        print(f"{name}: first activity {dt.strftime('%Y-%m-%d')}  (unix: {ts})")

rn1: first activity 2025-07-09  (unix: 1752062570)
sovereign: first activity 2025-08-07  (unix: 1754578485)
swisstony: first activity 2025-08-09  (unix: 1754716147)


## Full activity backfill with time-window chunking

In [9]:
MIN_WINDOW_SECONDS = 3600  # 1 hour floor — prevents infinite recursion

In [10]:
def save_chunk(records, wallet_name):
    """
    Append a batch of records to the wallet's parquet file.
    Deduplicates using transactionHash when available, falling back
    to a composite key for events that have empty transactionHash
    (e.g. some SPLIT/MERGE events).
    """
    if not records:
        return

    def make_dedup_key(row):
        # transactionHash is reliable for TRADE and REDEEM.
        # SPLIT/MERGE may return empty string — use composite fallback.
        if row["transactionHash"]:
            return row["transactionHash"]
        return f"{row['timestamp']}_{row['conditionId']}_{row['type']}"

    path = f"{DATA_DIR}/activity_{wallet_name}.parquet"
    df   = pd.DataFrame(records)
    df["_dedup_key"] = df.apply(make_dedup_key, axis=1)

    if os.path.exists(path):
        existing = pd.read_parquet(path, engine="fastparquet")
        existing["_dedup_key"] = existing.apply(make_dedup_key, axis=1)
        combined = pd.concat([existing, df], ignore_index=True)
        combined = combined.drop_duplicates(subset=["_dedup_key"])
        combined = combined.drop(columns=["_dedup_key"])
        combined.to_parquet(path, index=False, engine="fastparquet")
    else:
        df = df.drop_duplicates(subset=["_dedup_key"])
        df = df.drop(columns=["_dedup_key"])
        df.to_parquet(path, index=False, engine="fastparquet")

In [11]:
def fetch_window(address, win_start, win_end):
    """
    Fetch all records in a single time window via offset pagination.

    Returns (records, hit_cap).
    hit_cap=True means the window is too dense and should be split.
    We stop at offset 3000 (not 3500) to stay safely under the hard cap.
    """
    all_records = {}
    offset      = 0
    limit       = 500
    SAFE_CAP    = 3000

    while True:
        if offset > SAFE_CAP:
            return list(all_records.values()), True

        batch = get(DATA_API, "/activity", params={
            "user":          address,
            "start":         win_start,
            "end":           win_end,
            "limit":         limit,
            "offset":        offset,
            "sortDirection": "ASC",
        })

        if not batch:
            break

        for record in batch:
            key = record.get("transactionHash") or f"{record['timestamp']}_{record['conditionId']}"
            all_records[key] = record

        if len(batch) < limit:
            break
        
        time.sleep(0.5)
        offset += limit

    return list(all_records.values()), False

In [12]:
def fetch_recursive(address, win_start, win_end, depth=0):
    """
    Recursively fetch all activity in [win_start, win_end].

    If the window overflows the offset cap, bisect it and
    recurse into each half. Continues until windows fit
    or the minimum window size is reached.

    depth is tracked only for readable indentation in logs.
    """
    indent = "  " * (depth + 1)
    label  = datetime.fromtimestamp(win_start, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
    span   = win_end - win_start

    records, hit_cap = fetch_window(address, win_start, win_end)

    if not hit_cap:
        print(f"{indent}{label} (+{span//3600}h) | +{len(records)} records")
        return records

    # Window overflowed — check if we can still bisect
    if span <= MIN_WINDOW_SECONDS:
        print(f"{indent}⚠️  {label} | minimum window reached with overflow — data may be incomplete")
        return records

    # Bisect and recurse
    mid = (win_start + win_end) // 2
    print(f"{indent}{label} | overflow → bisecting into two {span//2//3600}h windows")

    left  = fetch_recursive(address, win_start, mid, depth + 1)
    right = fetch_recursive(address, mid, win_end, depth + 1)

    # Deduplicate across halves (bisect boundary could duplicate a record)
    merged = {}
    for record in left + right:
        key = record.get("transactionHash") or f"{record['timestamp']}_{record['conditionId']}"
        merged[key] = record

    return list(merged.values())

In [13]:
def fetch_activity_full(address, wallet_name, start_ts, end_ts, ledger, initial_window_days=7):
    """
    Fetch complete activity history, with per-window checkpointing.
    Skips any window already recorded in the ledger.
    Appends each completed window to the wallet's parquet file immediately.
    """
    current = start_ts
    delta   = initial_window_days * 86400
    total   = 0

    while current < end_ts:
        win_end = min(current + delta, end_ts)

        if window_done(ledger, wallet_name, current, win_end):
            print(f"  skip {datetime.fromtimestamp(current, tz=timezone.utc).strftime('%Y-%m-%d')} (already fetched)")
            current = win_end
            continue

        # Fetch this window (with recursive bisection if needed)
        records = fetch_recursive(address, current, win_end, depth=0)

        # Checkpoint: write to disk THEN mark ledger
        save_chunk(records, wallet_name)
        mark_window_done(ledger, wallet_name, current, win_end)

        total  += len(records)
        current = win_end

    return total

In [14]:
# Full history from each wallet's first known activity
WALLET_FIRST_TS = {
    "rn1":       1752062570,   # 2025-07-09
    "sovereign":  1754578485,   # 2025-08-07
    "swisstony":  1754716147,   # 2025-08-09
}

NOW_TS = int(datetime.now(tz=timezone.utc).timestamp())

In [ ]:
ledger = load_ledger()

for name, address in WALLETS.items():
    now_ts   = int(datetime.now(tz=timezone.utc).timestamp())  # fresh each wallet
    start_dt = datetime.fromtimestamp(WALLET_FIRST_TS[name], tz=timezone.utc).strftime("%Y-%m-%d")
    print(f"\n{'='*50}")
    print(f"Fetching {name} from {start_dt}")
    total = fetch_activity_full(address, name, WALLET_FIRST_TS[name], now_ts, ledger)
    print(f"→ {name}: +{total} new records this run")


Fetching rn1 from 2025-07-09
  skip 2025-07-09 (already fetched)
  skip 2025-07-16 (already fetched)
  skip 2025-07-23 (already fetched)
  skip 2025-07-30 (already fetched)
  skip 2025-08-06 (already fetched)
  skip 2025-08-13 (already fetched)
  skip 2025-08-20 (already fetched)
  skip 2025-08-27 (already fetched)
  skip 2025-09-03 (already fetched)
  skip 2025-09-10 (already fetched)
  skip 2025-09-17 (already fetched)
  skip 2025-09-24 (already fetched)
  skip 2025-10-01 (already fetched)
  skip 2025-10-08 (already fetched)
  skip 2025-10-15 (already fetched)
  skip 2025-10-22 (already fetched)
  skip 2025-10-29 (already fetched)
  skip 2025-11-05 (already fetched)
  skip 2025-11-12 (already fetched)
  skip 2025-11-19 (already fetched)
  skip 2025-11-26 (already fetched)
  skip 2025-12-03 (already fetched)
  skip 2025-12-10 (already fetched)
  skip 2025-12-17 (already fetched)
  skip 2025-12-24 (already fetched)
  skip 2025-12-31 (already fetched)
  skip 2026-01-07 (already fetched